# S6E10 — 01 Final Features (X_final)

**Purpose:** rebuild the frozen feature set in one fast run (~5 min, CPU) and save it, so that every later notebook
(Optuna, CatBoost/XGB, MLP, stack) just loads parquet files instead of repeating old experiments.

**X_final =** 21 baseline columns + 2 frequency features + route profile (`fdm_*`) + original-model signal (`orig_prob`, `orig_logit`).
`te_flight_distance` is NOT saved: target encoding must be computed inside each fold (helper in the last cell).

**Reference numbers (must be reproduced):** orig-model AUC on train ≈ 0.95472; tuned LGBM on X_final: CV 0.96072, LB 0.96036.

Outputs (`/kaggle/working`): `X_final_train.parquet`, `X_final_test.parquet`, `y_train.parquet`.

## 1. Setup

In [ ]:
import os, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e10') if Path('/kaggle/input/competitions/playground-series-s6e10').exists() \
           else next((p for p in (Path('data/raw'), Path('../data/raw')) if p.exists()), Path('data/raw'))
ORIG_PATH = next((p for p in [Path('/kaggle/input/datasets/deonissx/s6e10-orig-data/orig_data.csv'),
                              Path('data/raw/orig_data.csv'), Path('../data/raw/orig_data.csv')] if p.exists()),
                 Path('data/raw/orig_data.csv'))   # Kaggle path first, then local project folder
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else next((p for p in (Path('data/processed'), Path('../data/processed')) if p.exists()), Path('data/processed'))
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
print('data:', DATA_DIR, '| orig exists:', ORIG_PATH.exists(), '| out:', OUT_DIR)

## 2. Load data, preprocessing, frequency features
Identical to `features.ipynb` / the modelling notebook.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
train.columns = train.columns.str.lower().str.replace(' ', '_')
test.columns  = test.columns.str.lower().str.replace(' ', '_')

# 1. missing-value flag BEFORE fillna
train['arrival_delay_missing'] = train['arrival_delay_in_minutes'].isna().astype(int)
test['arrival_delay_missing']  = test['arrival_delay_in_minutes'].isna().astype(int)
# 2. fill gaps in arrival_delay with departure_delay
train['arrival_delay_in_minutes'] = train['arrival_delay_in_minutes'].fillna(train['departure_delay_in_minutes'])
test['arrival_delay_in_minutes']  = test['arrival_delay_in_minutes'].fillna(test['departure_delay_in_minutes'])
# 3. target
y_train = train['satisfaction'].astype(int)
# 4. drop id / target / arrival_delay
CAT_COLS  = ['gender', 'customer_type', 'type_of_travel', 'class']
DROP_COLS = ['id', 'satisfaction', 'arrival_delay_in_minutes']
X_raw      = train.drop(columns=DROP_COLS)
X_test_raw = test.drop(columns=['id', 'arrival_delay_in_minutes'])

def add_features(train_df, test_df):
    """flight_distance_freq and flight_distance_type_of_travel_freq (counts over train+test, no target)."""
    train_df = train_df.copy(); test_df = test_df.copy()
    full = pd.concat([train_df, test_df], ignore_index=True); n = len(train_df)
    counts = full['flight_distance'].map(full['flight_distance'].value_counts())
    train_df['flight_distance_freq'] = counts.iloc[:n].values
    test_df['flight_distance_freq']  = counts.iloc[n:].values
    key = full['flight_distance'].astype(str) + '_' + full['type_of_travel'].astype(str)
    counts = key.map(key.value_counts())
    train_df['flight_distance_type_of_travel_freq'] = counts.iloc[:n].values
    test_df['flight_distance_type_of_travel_freq']  = counts.iloc[n:].values
    return train_df, test_df

X_train, X_test = add_features(X_raw, X_test_raw)
for col in CAT_COLS:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')
print(f'Train: {X_train.shape}, Test: {X_test.shape} | positive rate {y_train.mean():.3f}')

## 3. Original-dataset signal (`orig_prob`, `orig_logit`)
A regularized LightGBM trained **only on the original dataset**, then used to score competition rows.
No competition labels are involved, so there is no target leakage.

In [ ]:
orig = pd.read_csv(ORIG_PATH)
orig.columns = orig.columns.str.lower().str.replace(' ', '_')
feats = [c for c in train.columns if c in orig.columns and c not in ('id', 'satisfaction', 'arrival_delay_missing')]
print(orig.shape, '| positive rate:', round(orig['satisfaction'].astype(int).mean(), 3), '| features:', len(feats))

orig_f = orig.copy()
orig_f['arrival_delay_in_minutes'] = orig_f['arrival_delay_in_minutes'].fillna(orig_f['departure_delay_in_minutes'])
orig_y = orig_f['satisfaction'].astype(int)

orig_X  = orig_f[feats].copy()
tr_orig = train[feats].copy()
te_orig = test[feats].copy()
for col in CAT_COLS:
    cats = sorted(set(orig_X[col].astype(str)) | set(tr_orig[col].astype(str)) | set(te_orig[col].astype(str)))
    for df in (orig_X, tr_orig, te_orig):
        df[col] = pd.Categorical(df[col].astype(str), categories=cats)

m_orig = lgb.LGBMClassifier(
    objective='binary', n_estimators=600, learning_rate=0.05, max_depth=8,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.5,
    reg_lambda=5.0, reg_alpha=0.05, random_state=123, n_jobs=-1, verbosity=-1)
m_orig.fit(orig_X, orig_y, categorical_feature=CAT_COLS)

p_tr = m_orig.predict_proba(tr_orig)[:, 1]
p_te = m_orig.predict_proba(te_orig)[:, 1]
orig_auc = roc_auc_score(y_train, p_tr)
print('AUC of the orig-model on competition train:', round(orig_auc, 5), '(reference 0.95472)')
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1 - 1e-6))

## 4. Route profile (`fdm_*`)
`flight_distance` behaves like a route ID. For each distance value: the mean of every other feature
(categoricals as codes), computed over train+test together. No target involved.

In [ ]:
full = pd.concat([X_train, X_test], ignore_index=True)
n = len(X_train)
prof_cols = [c for c in X_train.columns
             if c not in ('flight_distance', 'flight_distance_freq', 'flight_distance_type_of_travel_freq')]
coded = full[prof_cols].copy()
for c in CAT_COLS:
    coded[c] = coded[c].astype('category').cat.codes
coded['flight_distance'] = full['flight_distance'].values
grp = coded.groupby('flight_distance')
PROF = pd.DataFrame({f'fdm_{c}': grp[c].transform('mean').astype('float32') for c in prof_cols})
print('profile columns:', PROF.shape[1])

## 5. Assemble X_final and save

In [ ]:
X_final = X_train.copy()
X_test_final = X_test.copy()
for c in PROF.columns:
    X_final[c]      = PROF[c].values[:n]
    X_test_final[c] = PROF[c].values[n:]
X_final['orig_prob'], X_final['orig_logit']           = p_tr, logit(p_tr)
X_test_final['orig_prob'], X_test_final['orig_logit'] = p_te, logit(p_te)
for col in CAT_COLS:
    X_final[col]      = X_final[col].astype('category')
    X_test_final[col] = X_test_final[col].astype('category')

# --- sanity checks ---
assert list(X_final.columns) == list(X_test_final.columns)
assert X_final.shape[0] == 699635 and X_test_final.shape[0] == 299844, 'unexpected row counts'
assert not X_final.isna().any().any() and not X_test_final.isna().any().any(), 'NaN in features'
assert abs(orig_auc - 0.95472) < 0.0010, f'orig-model AUC {orig_auc:.5f} differs from reference 0.95472'
print('X_final:', X_final.shape, '| X_test_final:', X_test_final.shape)

X_final.to_parquet(OUT_DIR / 'X_final_train.parquet')
X_test_final.to_parquet(OUT_DIR / 'X_final_test.parquet')
y_train.reset_index(drop=True).to_frame('y').to_parquet(OUT_DIR / 'y_train.parquet')
print('saved to', OUT_DIR, '->', sorted(p.name for p in OUT_DIR.glob('*.parquet')))

## 6. (Optional) Reproduction check — tuned LightGBM on X_final
Set `RUN_CHECK = True` once (~25–40 min on CPU) to confirm the rebuilt features match the earlier ones:
expected **CV ≈ 0.96072** (folds 0.96095 / 0.95984 / 0.96153 / 0.96080 / 0.96057).
Also contains the in-fold target-encoding helper reused by later notebooks.

In [ ]:
RUN_CHECK = False

def add_te_flight_distance(X_fit, y_fit, other_frames, smooth='auto'):
    """te_flight_distance WITHOUT leakage. Call INSIDE a fold:
    X_fit/y_fit = training part of the fold; other_frames = validation part, test set."""
    enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=RANDOM_STATE)
    key = lambda d: d[['flight_distance']].astype(str)
    X_fit = X_fit.copy()
    X_fit['te_flight_distance'] = enc.fit_transform(key(X_fit), y_fit)[:, 0]
    encoded = []
    for frame in other_frames:
        frame = frame.copy()
        frame['te_flight_distance'] = enc.transform(key(frame))[:, 0]
        encoded.append(frame)
    return X_fit, encoded

LGBM_OPT_PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'verbose': -1, 'random_state': RANDOM_STATE,
    'n_estimators': 1000, 'early_stopping_rounds': 50,
    'learning_rate': 0.011594054946533571, 'num_leaves': 232, 'min_child_samples': 69,
    'subsample': 0.9851580134791578, 'colsample_bytree': 0.6505963546420028,
    'reg_alpha': 1.5621237403785044, 'reg_lambda': 1.7672411530478345,
}

if RUN_CHECK:
    oof = np.zeros(len(X_final))
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_final, y_train), 1):
        X_tr, X_val = X_final.iloc[tr_idx], X_final.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        X_tr, (X_val,) = add_te_flight_distance(X_tr, y_tr, [X_val])
        model = lgb.LGBMClassifier(**LGBM_OPT_PARAMS)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=CAT_COLS)
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
        print(f'Fold {fold} AUC: {roc_auc_score(y_val, oof[val_idx]):.5f}')
    print(f'CV AUC (OOF): {roc_auc_score(y_train, oof):.5f}  (expected ~0.96072)')